# ORIGINVAULT — Kronos model smoke test\nThis notebook tests Kronos-small in an interactive Colab runtime. It is for development/testing, not a permanent API host.

In [ ]:
!pip -q install pandas numpy torch requests\n!git clone --depth 1 https://github.com/shiyu-coder/Kronos.git\n%cd Kronos\n!pip -q install -r requirements.txt

In [ ]:
import torch, pandas as pd, numpy as np, requests\nfrom model import Kronos, KronosTokenizer, KronosPredictor\ndevice = 'cuda:0' if torch.cuda.is_available() else 'cpu'\nprint('Runtime device:', device)\ntokenizer = KronosTokenizer.from_pretrained('NeoQuasar/Kronos-Tokenizer-base')\nmodel = Kronos.from_pretrained('NeoQuasar/Kronos-small')\npredictor = KronosPredictor(model, tokenizer, device=device, max_context=512)

In [ ]:
symbol='BTCUSDT'; interval='15m'; limit=300\nurl=f'https://data-api.binance.vision/api/v3/klines?symbol={symbol}&interval={interval}&limit={limit}'\nraw=requests.get(url, timeout=20); raw.raise_for_status()\ncols=['time','open','high','low','close','volume','close_time','quote_volume','trades','taker_base','taker_quote','ignore']\ndata=pd.DataFrame(raw.json(),columns=cols)\nfor col in ['open','high','low','close','volume']: data[col]=data[col].astype(float)\ndata['time']=pd.to_datetime(data['time'],unit='ms')\nx_df=data[['open','high','low','close','volume']].copy(); x_ts=data['time']\nhorizon=12; y_ts=pd.date_range(x_ts.iloc[-1],periods=horizon+1,freq='15min')[1:]\nforecast=predictor.predict(df=x_df,x_timestamp=x_ts,y_timestamp=y_ts,pred_len=horizon,T=1.0,top_p=0.9,sample_count=1)\ndisplay(forecast.head()); print('Last observed close:',x_df.close.iloc[-1]); print('Last forecast close:',forecast.close.iloc[-1])

## Notes\nRepeat `predictor.predict` several times to create a sample-path range. P10–P90 describes the generated sample distribution only; it is not a calibrated probability interval. Do not use Colab as a public always-on web service.